# 🧹 02 — Preprocessing Demo

Test từng bước preprocessing trên sample data trước khi chạy trên Spark.

**Pipeline:** Raw text → Remove boilerplate → Clean → Tokenize → Remove stopwords

## 2.1 Setup

In [1]:
import os, sys, re

# Ensure project root is on path
os.chdir(os.path.join(os.path.dirname(os.getcwd()) if 'notebooks' in os.getcwd() else os.getcwd()))
sys.path.insert(0, '.')

from config.settings import config
print(f"Environment: {config.ENV}")
print(f"Raw data: {config.DATA_RAW_PATH}")
print(f"Cleaned output: {config.DATA_CLEANED_PATH}")

Environment: dev
Raw data: ./data/sample/
Cleaned output: ./data/output/cleaned/


## 2.2 Load a sample book

In [2]:
import glob

sample_files = sorted(glob.glob(config.DATA_RAW_PATH + "*.txt"))
print(f"Found {len(sample_files)} books in {config.DATA_RAW_PATH}")

# Load one sample book
sample_path = sample_files[0]
with open(sample_path, 'r', encoding='utf-8') as f:
    raw_text = f.read()

book_name = os.path.basename(sample_path)
print(f"\n--- {book_name} ({len(raw_text):,} chars) ---")
print(raw_text[:1000])

Found 93 books in ./data/sample/

--- pg10007.txt (154,333 chars) ---
Carmilla by Joseph Sheridan Le Fanu Copyright 1872 Contents PROLOGUE CHAPTER I. An Early Fright CHAPTER II. A Guest CHAPTER III. We Compare Notes CHAPTER IV. Her Habits A Saunter CHAPTER V. A Wonderful Likeness CHAPTER VI. A Very Strange Agony CHAPTER VII. Descending CHAPTER VIII. Search CHAPTER IX. The Doctor CHAPTER X. Bereaved CHAPTER XI. The Story CHAPTER XII. A Petition CHAPTER XIII. The Woodman CHAPTER XIV. The Meeting CHAPTER XV. Ordeal and Execution CHAPTER XVI. Conclusion PROLOGUE Upon a paper attached to the Narrative which follows, Doctor Hesselius has written a rather elaborate note, which he accompanies with a reference to his Essay on the strange subject which the MS. illuminates. This mysterious subject he treats, in that Essay, with his usual learning and acumen, and with remarkable directness and condensation. It will form but one volume of the series of that extraordinary man's collected papers. As 

## 2.3 Step 1 — Remove Gutenberg boilerplate

In [3]:
from scripts.text_cleaning_utils import strip_gutenberg_headers

cleaned = strip_gutenberg_headers(raw_text)
print(f"Before: {len(raw_text):,} chars → After: {len(cleaned):,} chars")
print(f"Removed: {len(raw_text) - len(cleaned):,} chars of Gutenberg boilerplate")
print(f"\n--- First 500 chars after stripping ---")
print(cleaned[:500])

Before: 154,333 chars → After: 154,333 chars
Removed: 0 chars of Gutenberg boilerplate

--- First 500 chars after stripping ---
Carmilla by Joseph Sheridan Le Fanu Copyright 1872 Contents PROLOGUE CHAPTER I. An Early Fright CHAPTER II. A Guest CHAPTER III. We Compare Notes CHAPTER IV. Her Habits A Saunter CHAPTER V. A Wonderful Likeness CHAPTER VI. A Very Strange Agony CHAPTER VII. Descending CHAPTER VIII. Search CHAPTER IX. The Doctor CHAPTER X. Bereaved CHAPTER XI. The Story CHAPTER XII. A Petition CHAPTER XIII. The Woodman CHAPTER XIV. The Meeting CHAPTER XV. Ordeal and Execution CHAPTER XVI. Conclusion PROLOGUE Upon 


## 2.4 Step 2 — Lowercase + regex clean

In [4]:
# Lowercase + remove non-alpha (keep letters and spaces only)
lowered = cleaned.lower()
alpha_only = re.sub(r"[^a-z\s]", "", lowered)

print(f"Sample before cleaning:\n  {cleaned[200:400]}")
print(f"\nSample after lowercase + regex:\n  {alpha_only[200:400]}")

Sample before cleaning:
  l Likeness CHAPTER VI. A Very Strange Agony CHAPTER VII. Descending CHAPTER VIII. Search CHAPTER IX. The Doctor CHAPTER X. Bereaved CHAPTER XI. The Story CHAPTER XII. A Petition CHAPTER XIII. The Wood

Sample after lowercase + regex:
  s chapter vi a very strange agony chapter vii descending chapter viii search chapter ix the doctor chapter x bereaved chapter xi the story chapter xii a petition chapter xiii the woodman chapter xiv t


## 2.5 Step 3 — Tokenization

In [5]:
# Tokenize by splitting on whitespace
raw_tokens = alpha_only.split()
print(f"Total tokens: {len(raw_tokens):,}")
print(f"First 30 tokens: {raw_tokens[:30]}")
print(f"Unique tokens: {len(set(raw_tokens)):,}")

Total tokens: 28,185
First 30 tokens: ['carmilla', 'by', 'joseph', 'sheridan', 'le', 'fanu', 'copyright', 'contents', 'prologue', 'chapter', 'i', 'an', 'early', 'fright', 'chapter', 'ii', 'a', 'guest', 'chapter', 'iii', 'we', 'compare', 'notes', 'chapter', 'iv', 'her', 'habits', 'a', 'saunter', 'chapter']
Unique tokens: 4,053


## 2.6 Step 4 — Stopword removal

In [6]:
from src.preprocessing import _ensure_nltk_stopwords

stop_set = _ensure_nltk_stopwords()
print(f"NLTK English stopwords: {len(stop_set)} words")
print(f"Examples: {sorted(list(stop_set))[:20]}")

# Filter: remove stopwords and single-char tokens
final_tokens = [t for t in raw_tokens if t not in stop_set and len(t) > 1]
removed = len(raw_tokens) - len(final_tokens)

print(f"\nBefore: {len(raw_tokens):,} tokens → After: {len(final_tokens):,} tokens")
print(f"Removed: {removed:,} tokens ({removed/len(raw_tokens)*100:.1f}%)")
print(f"First 30 final tokens: {final_tokens[:30]}")

NLTK English stopwords: 198 words
Examples: ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been']

Before: 28,185 tokens → After: 12,957 tokens
Removed: 15,228 tokens (54.0%)
First 30 final tokens: ['carmilla', 'joseph', 'sheridan', 'le', 'fanu', 'copyright', 'contents', 'prologue', 'chapter', 'early', 'fright', 'chapter', 'ii', 'guest', 'chapter', 'iii', 'compare', 'notes', 'chapter', 'iv', 'habits', 'saunter', 'chapter', 'wonderful', 'likeness', 'chapter', 'vi', 'strange', 'agony', 'chapter']


## 2.7 Test with PySpark (local mode)

In [7]:
from src.preprocessing import create_spark_session, load_raw_books, preprocess_books

# Create SparkSession (uses config: local[*], 4g driver memory)
spark = create_spark_session()
spark.sparkContext.setLogLevel("ERROR")

# Load all sample books
raw_df = load_raw_books(spark)
print(f"Loaded {raw_df.count()} raw books")

# Run full preprocessing pipeline
result_df = preprocess_books(raw_df, spark)
print(f"Preprocessed: {result_df.count()} books")
print(f"\nSchema:")
result_df.printSchema()
result_df.show(10, truncate=80)

26/03/03 12:10:43 WARN Utils: Your hostname, MacBook-Air-cua-Tran.local resolves to a loopback address: 127.0.0.1; using 172.20.10.10 instead (on interface en0)
26/03/03 12:10:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/03 12:10:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Loaded 93 raw books


Preprocessed: 93 books

Schema:
root
 |-- book_id: string (nullable = true)
 |-- tokens: array (nullable = true)
 |    |-- element: string (containsNull = true)



+-------+--------------------------------------------------------------------------------+
|book_id|                                                                          tokens|
+-------+--------------------------------------------------------------------------------+
|pg10007|[carmilla, joseph, sheridan, le, fanu, copyright, contents, prologue, chapter...|
| pg1026|[diary, nobody, george, grossmith, weedon, grossmith, illustrations, weedon, ...|
|  pg103|[illustration, around, world, eighty, days, jules, verne, contents, chapter, ...|
|pg10444|[peace, negotiations, personal, narrative, robert, lansing, illustrations, co...|
|pg10554|[right, ho, jeeves, wodehouse, raymond, needham, kc, affection, admiration, j...|
|pg11592|[note, project, gutenberg, also, html, version, file, includes, original, lov...|
| pg1184|[count, monte, cristo, alexandre, dumas, pre, contents, volume, one, chapter,...|
|pg12027|[crime, cause, treatment, clarence, darrow, preface, book, comes, reflections...|

## 2.8 Verify Parquet output

In [8]:
# Save as Parquet and read back to verify
from pyspark.sql import functions as F

result_df.coalesce(1).write.mode("overwrite").parquet(config.DATA_CLEANED_PATH)

# Read back and verify
parquet_df = spark.read.parquet(config.DATA_CLEANED_PATH)
print(f"Parquet row count: {parquet_df.count()}")
print(f"Schema: {parquet_df.schema.simpleString()}")

# Show token count distribution
parquet_df.withColumn("num_tokens", F.size("tokens")) \
    .select("book_id", "num_tokens") \
    .orderBy(F.desc("num_tokens")) \
    .show(10)

Parquet row count: 93
Schema: struct<book_id:string,tokens:array<string>>


+-------+----------+
|book_id|num_tokens|
+-------+----------+
| pg2760|    295803|
| pg1184|    220057|
|pg14287|    186905|
|pg54873|    134576|
| pg5097|    133018|
|pg24790|    114958|
| pg1468|    110959|
| pg1320|    109784|
| pg1257|    108884|
| pg3317|     90704|
+-------+----------+
only showing top 10 rows



## 2.9 Cleanup

In [9]:
spark.stop()
print("SparkSession stopped.")

SparkSession stopped.


---
## Kết luận

- [x] Preprocessing functions hoạt động đúng trên sample
- [x] Gutenberg headers/footers được remove chính xác
- [x] Parquet output có đúng schema `(book_id: string, tokens: array<string>)`
- [x] Ready to run on full dataset

### Cách sử dụng nhanh

```python
# Option 1: Full pipeline (load → clean → save Parquet)
from src.preprocessing import run_preprocessing
df = run_preprocessing()  # Saves to data/output/cleaned/

# Option 2: Step-by-step
from src.preprocessing import create_spark_session, load_raw_books, preprocess_books
spark = create_spark_session()
raw_df = load_raw_books(spark)
result_df = preprocess_books(raw_df, spark)
# result_df has columns: book_id (string), tokens (array<string>)

# Option 3: Just the text cleaning (no Spark)
from scripts.text_cleaning_utils import strip_gutenberg_headers
clean_text = strip_gutenberg_headers(raw_text)
```

### Next: Shingling
Notebook `03_lsh_pipeline_demo.ipynb` sẽ consume Parquet output từ `data/output/cleaned/`.